# 04 — Highlighting Intersected Features

Knowing *which* countries a path crosses is useful data. Seeing it on a map is useful communication.

This notebook takes the output of `find_crossed_countries` and translates it into a two-layer map: one layer for **hit** countries (highlighted), one for **everything else** (dimmed). The visual contrast makes the result unambiguous at a glance.

## Setup

In [2]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "paths.geojson") as f:
    module_paths = json.load(f)

with open(DATA_DIR / "countries.geojson") as f:
    countries_fc = json.load(f)


# All helpers from previous notebooks
def orientation(p, q, r):
    return (q[0] - p[0]) * (r[1] - q[1]) - (q[1] - p[1]) * (r[0] - q[0])

def on_segment(p, q, r):
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def segments_intersect(p1, p2, p3, p4):
    d1 = orientation(p3, p4, p1)
    d2 = orientation(p3, p4, p2)
    d3 = orientation(p1, p2, p3)
    d4 = orientation(p1, p2, p4)
    if (d1 > 0 and d2 < 0 or d1 < 0 and d2 > 0) and \
       (d3 > 0 and d4 < 0 or d3 < 0 and d4 > 0):
        return True
    if d1 == 0 and on_segment(p3, p1, p4): return True
    if d2 == 0 and on_segment(p3, p2, p4): return True
    if d3 == 0 and on_segment(p1, p3, p2): return True
    if d4 == 0 and on_segment(p1, p4, p2): return True
    return False

def polygon_edges(geometry):
    if geometry["type"] == "Polygon":
        rings = geometry["coordinates"]
    elif geometry["type"] == "MultiPolygon":
        rings = [ring for poly in geometry["coordinates"] for ring in poly]
    else:
        return []
    return [(ring[i], ring[i+1]) for ring in rings for i in range(len(ring)-1)]

def segment_bbox(p1, p2):
    return (min(p1[0], p2[0]), min(p1[1], p2[1]),
            max(p1[0], p2[0]), max(p1[1], p2[1]))

def geometry_bbox(geometry):
    rings = geometry["coordinates"] if geometry["type"] == "Polygon" \
            else [ring for poly in geometry["coordinates"] for ring in poly]
    coords = [pt for ring in rings for pt in ring]
    return (min(c[0] for c in coords), min(c[1] for c in coords),
            max(c[0] for c in coords), max(c[1] for c in coords))

def bboxes_overlap(b1, b2):
    return not (b1[2] < b2[0] or b2[2] < b1[0] or
                b1[3] < b2[1] or b2[3] < b1[1])

def path_crosses_feature(path_feature, country_feature):
    coords = path_feature["geometry"]["coordinates"]
    geom   = country_feature["geometry"]
    if geom is None:
        return False
    country_box = geometry_bbox(geom)
    edges = polygon_edges(geom)
    for i in range(len(coords) - 1):
        p1, p2 = coords[i], coords[i + 1]
        if not bboxes_overlap(segment_bbox(p1, p2), country_box):
            continue
        for e1, e2 in edges:
            if segments_intersect(p1, p2, e1, e2):
                return True
    return False

def find_crossed_countries(path_feature, countries_fc):
    return [c for c in countries_fc["features"] if path_crosses_feature(path_feature, c)]

if isinstance(countries_fc, list):
    countries_fc = {"type": "FeatureCollection", "features": countries_fc}

print(f"Loaded {len(module_paths['features'])} paths, {len(countries_fc['features'])} countries.")

Loaded 4 paths, 255 countries.


## Separating Hit vs. Not-Hit Features

Once we have the list of crossed countries, splitting the full dataset into two groups is a straightforward filter. We use a **set of names** for O(1) membership checks rather than searching the crossed list on every iteration.

In [7]:
# Work with the Alpha path: Washington D.C. → Tehran
alpha = next(f for f in module_paths["features"] if f["properties"]["name"] == "Alpha")

crossed  = find_crossed_countries(alpha, countries_fc)
# Work with the Alpha path: Washington D.C. → Tehran
alpha = next(f for f in module_paths["features"] if f["properties"]["name"] == "Alpha")

crossed  = find_crossed_countries(alpha, countries_fc)
hit_names = {c["properties"].get("name") for c in crossed if c["properties"].get("name") is not None}

hit_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"]
                 if c["properties"].get("name") in hit_names]
}
miss_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"]
                 if c["properties"].get("name") not in hit_names]
}

print(f"Alpha path:  {len(hit_fc['features'])} hit,  {len(miss_fc['features'])} not hit")
print("Crossed:", sorted(hit_names))

hit_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"] if c["properties"].get("name") in hit_names]
}
miss_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"] if c["properties"].get("name") not in hit_names]
}

print(f"Alpha path:  {len(hit_fc['features'])} hit,  {len(miss_fc['features'])} not hit")
print("Crossed:", sorted(hit_names))

# Work with the Alpha path: Washington D.C. → Tehran
alpha = next(f for f in module_paths["features"] if f["properties"]["name"] == "Alpha")

crossed = find_crossed_countries(alpha, countries_fc)
hit_names = {c["properties"].get("name") for c in crossed if c["properties"].get("name") is not None}

hit_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"]
                 if c["properties"].get("name") in hit_names]
}
miss_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"]
                 if c["properties"].get("name") not in hit_names]
}

print(f"Alpha path:  {len(hit_fc['features'])} hit,  {len(miss_fc['features'])} not hit")
print("Crossed:", sorted(hit_names))
miss_fc = {
    "type": "FeatureCollection",
    "features": [c for c in countries_fc["features"]
                 if c["properties"].get("name") not in hit_names]
}

print(f"Alpha path:  {len(hit_fc['features'])} hit,  {len(miss_fc['features'])} not hit")
print("Crossed:", sorted(hit_names))

Alpha path:  0 hit,  255 not hit
Crossed: []
Alpha path:  0 hit,  255 not hit
Crossed: []
Alpha path:  0 hit,  255 not hit
Crossed: []
Alpha path:  0 hit,  255 not hit
Crossed: []


## Visual Feedback Principles

A good highlight layer uses **contrast**, not just color:

| Layer | Color | Fill opacity | Weight | Effect |
|---|---|---|---|---|
| Hit countries | `#e74c3c` (red) | 0.55 | 2 | Clearly stands out |
| Other countries | `#7f8c8d` (gray) | 0.15 | 0.5 | Present but recessive |
| Path | `#e74c3c` | — | 3 | Matches hit color |

The dimmed background lets the eye confirm which countries were *not* hit — as important as knowing which ones were.

In [8]:
# Two-layer map: hit countries highlighted, others dimmed
path_fc = {"type": "FeatureCollection", "features": [alpha]}

m = Map(center=(40, 20), zoom=3, basemap=basemaps.CartoDB.Positron)

m.add(GeoJSON(
    data=miss_fc,
    name="Other countries",
    style={"color": "#7f8c8d", "fillColor": "#7f8c8d",
           "fillOpacity": 0.15, "weight": 0.5}
))
m.add(GeoJSON(
    data=hit_fc,
    name="Crossed countries",
    style={"color": "#e74c3c", "fillColor": "#e74c3c",
           "fillOpacity": 0.55, "weight": 2}
))
m.add(GeoJSON(
    data=path_fc,
    name="Alpha path",
    style={"color": "#e74c3c", "weight": 3, "opacity": 0.9}
))
m.add(LayersControl())
m

Map(center=[40, 20], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Aggregate View: All Paths Combined

Highlight every country crossed by **any** path in the dataset. This gives an aggregate impact view — useful when analyzing a scenario with multiple simultaneous launches.

In [9]:
# Union of all countries crossed by any path
all_hit_names = set()
for path in module_paths["features"]:
    crossed = find_crossed_countries(path, countries_fc)
    all_hit_names.update(c["properties"]["name"] for c in crossed)

all_hit_fc  = {"type": "FeatureCollection",
               "features": [c for c in countries_fc["features"]
                             if c["properties"]["name"] in all_hit_names]}
all_miss_fc = {"type": "FeatureCollection",
               "features": [c for c in countries_fc["features"]
                             if c["properties"]["name"] not in all_hit_names]}

print(f"{len(all_hit_names)} unique countries crossed across all paths")

m2 = Map(center=(30, 20), zoom=2, basemap=basemaps.CartoDB.Positron)
m2.add(GeoJSON(
    data=all_miss_fc,
    style={"color": "#7f8c8d", "fillColor": "#7f8c8d",
           "fillOpacity": 0.1, "weight": 0.5}
))
m2.add(GeoJSON(
    data=all_hit_fc,
    style={"color": "#e67e22", "fillColor": "#e67e22",
           "fillOpacity": 0.5, "weight": 1.5}
))
m2.add(GeoJSON(
    data=module_paths,
    style={"color": "#e74c3c", "weight": 2, "opacity": 0.85}
))
m2

KeyError: 'name'

## Exercise A

The **Charlie** path runs from Caracas, Venezuela to Madrid, Spain.

1. Find the countries Charlie crosses.
2. Build a two-layer map: crossed countries in **red** (`#e74c3c`), others in **gray** (`#bdc3c7`), with the Charlie path drawn in red.
3. Print the count and names of crossed countries.

In [10]:
# Get the Charlie path from module_paths
charlie = next(
    f for f in module_paths["features"]
    if f["properties"]["name"] == "Charlie"
)

# 1. Find crossed countries
crossed = find_crossed_countries(charlie, countries_fc)

hit_names = {
    c["properties"].get("name")
    for c in crossed
    if c["properties"].get("name") is not None
}

# Separate crossed and non-crossed countries
hit_fc = {
    "type": "FeatureCollection",
    "features": [
        c for c in countries_fc["features"]
        if c["properties"].get("name") in hit_names
    ]
}

miss_fc = {
    "type": "FeatureCollection",
    "features": [
        c for c in countries_fc["features"]
        if c["properties"].get("name") not in hit_names
    ]
}

# 3. Print count and names
print(f"Charlie path: {len(hit_names)} countries crossed")
print("Crossed countries:")
for name in sorted(hit_names):
    print(name)

# 2. Build a two-layer map
path_fc = {
    "type": "FeatureCollection",
    "features": [charlie]
}

m = Map(center=(35, -25), zoom=3, basemap=basemaps.CartoDB.Positron)

m.add(GeoJSON(
    data=miss_fc,
    name="Other countries",
    style={
        "color": "#bdc3c7",
        "fillColor": "#bdc3c7",
        "fillOpacity": 0.15,
        "weight": 0.5
    }
))

m.add(GeoJSON(
    data=hit_fc,
    name="Crossed countries",
    style={
        "color": "#e74c3c",
        "fillColor": "#e74c3c",
        "fillOpacity": 0.55,
        "weight": 2
    }
))

m.add(GeoJSON(
    data=path_fc,
    name="Charlie path",
    style={
        "color": "#e74c3c",
        "weight": 3,
        "opacity": 0.9
    }
))

m.add(LayersControl())

m

Charlie path: 0 countries crossed
Crossed countries:


Map(center=[35, -25], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

## Exercise B

Build a **per-path layer map** using `LayersControl` so each path and its corresponding hit countries are a named, toggleable pair.

Use a different highlight color for each path:

| Path | Color |
|---|---|
| Alpha | `#e74c3c` |
| Bravo | `#2980b9` |
| Charlie | `#27ae60` |
| Delta | `#8e44ad` |

Add a single dimmed base layer for all non-hit countries.

In [11]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

PATH_COLORS = {
    "Alpha":   "#e74c3c",
    "Bravo":   "#2980b9",
    "Charlie": "#27ae60",
    "Delta":   "#8e44ad",
}

# Collect every country hit by at least one path
all_hit_names = set()

path_hit_data = {}

for path in module_paths["features"]:
    path_name = path["properties"]["name"]
    
    crossed = find_crossed_countries(path, countries_fc)
    
    hit_names = {
        c["properties"].get("name")
        for c in crossed
        if c["properties"].get("name") is not None
    }
    
    path_hit_data[path_name] = hit_names
    all_hit_names.update(hit_names)


# Single dimmed base layer for countries not hit by any path
non_hit_fc = {
    "type": "FeatureCollection",
    "features": [
        c for c in countries_fc["features"]
        if c["properties"].get("name") not in all_hit_names
    ]
}


# Create map
m = Map(center=(35, 10), zoom=2, basemap=basemaps.CartoDB.Positron)


# Add non-hit countries once
m.add(GeoJSON(
    data=non_hit_fc,
    name="Non-hit countries",
    style={
        "color": "#bdc3c7",
        "fillColor": "#bdc3c7",
        "fillOpacity": 0.15,
        "weight": 0.5
    }
))


# Add one hit-country layer and one path layer per route
for path in module_paths["features"]:
    path_name = path["properties"]["name"]
    color = PATH_COLORS.get(path_name, "#000000")
    hit_names = path_hit_data[path_name]
    
    hit_fc = {
        "type": "FeatureCollection",
        "features": [
            c for c in countries_fc["features"]
            if c["properties"].get("name") in hit_names
        ]
    }
    
    path_fc = {
        "type": "FeatureCollection",
        "features": [path]
    }
    
    m.add(GeoJSON(
        data=hit_fc,
        name=f"{path_name} hit countries",
        style={
            "color": color,
            "fillColor": color,
            "fillOpacity": 0.45,
            "weight": 1.5
        }
    ))
    
    m.add(GeoJSON(
        data=path_fc,
        name=f"{path_name} path",
        style={
            "color": color,
            "weight": 3,
            "opacity": 0.9
        }
    ))


m.add(LayersControl())

m

Map(center=[35, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---

## Check Your Understanding

**1.** Why is visual feedback important in spatial analysis — couldn't you just read a list of country names?

**2.** How can poor styling choices hide correct results? Give a concrete example.

```python
# No code needed — answer in your own words
```

## Check Your Understanding

### 1.
Why is visual feedback important in spatial analysis — couldn't you just read a list of country names?

A list of country names only tells you the final result, but a map helps verify whether the spatial logic actually makes sense.

Visual feedback allows you to:
- Spot incorrect intersections
- Notice missing countries
- See the actual route shape
- Detect geometry or coordinate errors

For example, if a path supposedly crosses Brazil but the map clearly stays over the Atlantic Ocean, the visualization immediately reveals a problem that a text list would not.

Maps also make spatial relationships easier for humans to understand than raw coordinate data or country lists.

---

### 2.
How can poor styling choices hide correct results? Give a concrete example.

Poor styling can make correct data difficult or impossible to see.

For example:
- If crossed countries and non-crossed countries use very similar colors, the highlighted results may blend into the background.
- If the path line is too thin or too transparent, it may appear missing even though the geometry is correct.
- If polygon fill opacity is too high, countries can completely cover the path underneath.

Concrete example:

If crossed countries are colored light gray and non-crossed countries are also light gray, the analysis may look incorrect because users cannot visually distinguish the hit countries from the background layer.

## Next

In [05 — Applications: Missile Paths](./05-Applications_Missile_Paths.ipynb), we combine intersection detection with distance and bearing to produce a complete intelligence picture for each path — impacted countries, range, and direction in one report.